## 1. Data

Lending Club 2016?Q3??:https://www.lendingclub.com/info/download-data.action

??:http://kldavenport.com/lending-club-data-analysis-revisted-with-python/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
df = pd.read_csv("./LoanStats_2016Q3.csv",skiprows=1,low_memory=False)

In [ ]:
df.info()

In [ ]:
df.head(3)

## 2. Keep what we need

In [ ]:
# .ix[row slice, column slice] 
df.ix[:4,:7]

In [ ]:
df.drop('id',1,inplace=True)
df.drop('member_id',1,inplace=True)

In [ ]:
df.int_rate = pd.Series(df.int_rate).str.replace('%', '').astype(float)

In [ ]:
df.ix[:4,:7]

### Loan Amount Requested Verus the Funded Amount

In [ ]:
print (df.loan_amnt != df.funded_amnt).value_counts()

In [ ]:
df.query('loan_amnt != funded_amnt').head(5)

In [ ]:
df.dropna(axis=0, how='all',inplace=True)

In [ ]:
df.info()

In [ ]:
df.dropna(axis=1, how='all',inplace=True)

In [ ]:
df.info()

In [ ]:
df.ix[:5,8:15]

### emp_title: employment title

In [ ]:
print df.emp_title.value_counts().head()
print df.emp_title.value_counts().tail()
df.emp_title.unique().shape

In [ ]:
df.drop(['emp_title'],1, inplace=True)

In [ ]:
df.ix[:5,8:15]

### emp_length: employment length

In [ ]:
df.emp_length.value_counts()

In [ ]:
df.replace('n/a', np.nan,inplace=True)
df.emp_length.fillna(value=0,inplace=True)
df['emp_length'].replace(to_replace='[^0-9]+', value='', inplace=True, regex=True)
df['emp_length'] = df['emp_length'].astype(int)

In [ ]:
df.emp_length.value_counts()

### verification status:"Indicates if income was verified by LC, not verified, or if the income source was verified"

In [ ]:
df.verification_status.value_counts()

### Target: Loan Statuses

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
pd.unique(df['loan_status'].values.ravel())

In [ ]:
for col in df.select_dtypes(include=['object']).columns:
    print ("Column {} has {} unique instances".format( col, len(df[col].unique())) )

In [ ]:
# ?????????,unique
df.select_dtypes(include=['O']).describe().T.\
assign(missing_pct=df.apply(lambda x : (len(x)-x.count())/float(len(x))))

In [ ]:
df.revol_util = pd.Series(df.revol_util).str.replace('%', '').astype(float)

In [ ]:
# missing_pct
df.drop('desc',1,inplace=True)
df.drop('verification_status_joint',1,inplace=True)

In [ ]:
df.drop('zip_code',1,inplace=True)
df.drop('addr_state',1,inplace=True)
df.drop('earliest_cr_line',1,inplace=True)
df.drop('revol_util',1,inplace=True)
df.drop('purpose',1,inplace=True)
df.drop('title',1,inplace=True)
df.drop('term',1,inplace=True)
df.drop('issue_d',1,inplace=True)
# df.drop('',1,inplace=True)
# ???????
df.drop(['out_prncp','out_prncp_inv','total_pymnt',
         'total_pymnt_inv','total_rec_prncp', 'grade', 'sub_grade'] ,1, inplace=True)
df.drop(['total_rec_int','total_rec_late_fee',
         'recoveries','collection_recovery_fee',
         'collection_recovery_fee' ],1, inplace=True)
df.drop(['last_pymnt_d','last_pymnt_amnt',
         'next_pymnt_d','last_credit_pull_d'],1, inplace=True)
df.drop(['policy_code'],1, inplace=True)

In [ ]:
df.info()

In [ ]:
df.ix[:5,:10]

In [ ]:
df.ix[:5,10:21]

In [ ]:
print df.columns
print df.head(1).values
df.info()

In [ ]:
df.select_dtypes(include=['float']).describe().T.\
assign(missing_pct=df.apply(lambda x : (len(x)-x.count())/float(len(x))))

In [ ]:
df.drop('annual_inc_joint',1,inplace=True)
df.drop('dti_joint',1,inplace=True)

In [ ]:
df.select_dtypes(include=['int']).describe().T.\
assign(missing_pct=df.apply(lambda x : (len(x)-x.count())/float(len(x))))

### Target: Loan Statuses

In [ ]:
df['loan_status'].value_counts()
# .plot(kind='bar')

In [ ]:
df.loan_status.replace('Fully Paid', int(1),inplace=True)
df.loan_status.replace('Current', int(1),inplace=True)
df.loan_status.replace('Late (16-30 days)', int(0),inplace=True)
df.loan_status.replace('Late (31-120 days)', int(0),inplace=True)
df.loan_status.replace('Charged Off', np.nan,inplace=True)
df.loan_status.replace('In Grace Period', np.nan,inplace=True)
df.loan_status.replace('Default', np.nan,inplace=True)
# df.loan_status.astype('int')
df.loan_status.value_counts()

In [ ]:
# df.loan_status
df.dropna(subset=['loan_status'],inplace=True)

### Highly Correlated Data

In [ ]:
cor = df.corr()
cor.loc[:,:] = np.tril(cor, k=-1) # below main lower triangle of an array
cor = cor.stack()
cor[(cor > 0.55) | (cor < -0.55)]

In [ ]:
df.drop(['funded_amnt','funded_amnt_inv', 'installment'], axis=1, inplace=True)

## 2. Our Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn import ensemble
from sklearn.preprocessing import OneHotEncoder #https://ljalphabeta.gitbooks.io/python-/content/categorical_data.html

In [ ]:
Y = df.loan_status
X = df.drop('loan_status',1,inplace=False)

In [ ]:
print Y.shape
print sum(Y)

In [ ]:
X = pd.get_dummies(X)

In [ ]:
print X.columns
print X.head(1).values
X.info()

In [ ]:
X.fillna(0.0,inplace=True)
X.fillna(0,inplace=True)

### Train Data & Test Data

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=.3, random_state=123)

In [ ]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

In [ ]:
print y_train.value_counts()
print y_test.value_counts()

### Gradient Boosting Regression Tree

In [ ]:
# param_grid = {'learning_rate': [0.1, 0.05, 0.02, 0.01],
#               'max_depth': [1,2,3,4],
#               'min_samples_split': [50,100,200,400],
#               'n_estimators': [100,200,400,800]
#               }

param_grid = {'learning_rate': [0.1],
              'max_depth': [2],
              'min_samples_split': [50,100],
              'n_estimators': [100,200]
              }
# param_grid = {'learning_rate': [0.1],
#               'max_depth': [4],
#               'min_samples_leaf': [3],
#               'max_features': [1.0],
#               }

est = GridSearchCV(ensemble.GradientBoostingRegressor(),
                   param_grid, n_jobs=4, refit=True)

est.fit(x_train, y_train)

best_params = est.best_params_
print best_params

In [ ]:
print best_params

In [ ]:
%%time
est = ensemble.GradientBoostingRegressor(min_samples_split=50,n_estimators=300,
                                         learning_rate=0.1,max_depth=1, random_state=0,loss='ls').\
fit(x_train, y_train)

In [ ]:
est.score(x_test,y_test)

In [ ]:
%%time
est = ensemble.GradientBoostingRegressor(min_samples_split=50,n_estimators=100,
                                         learning_rate=0.1,max_depth=2, random_state=0,loss='ls').\
fit(x_train, y_train)

In [ ]:
est.score(x_test,y_test)

In [ ]:
def compute_ks(data):

    sorted_list = data.sort_values(['predict'], ascending=[True])

    total_bad = sorted_list['label'].sum(axis=None, skipna=None, level=None, numeric_only=None) / 3
    total_good = sorted_list.shape[0] - total_bad

    # print "total_bad = ", total_bad
    # print "total_good = ", total_good

    max_ks = 0.0
    good_count = 0.0
    bad_count = 0.0
    for index, row in sorted_list.iterrows():
        if row['label'] == 3:
            bad_count += 1.0
        else:
            good_count += 1.0

        val = bad_count/total_bad - good_count/total_good
        max_ks = max(max_ks, val)

    return max_ks

In [ ]:
test_pd = pd.DataFrame()
test_pd['predict'] = est.predict(x_test)
test_pd['label'] = y_test
# df['predict'] = est.predict(x_test)
print compute_ks(test_pd[['label','predict']])

In [ ]:
# Top Ten
feature_importance = est.feature_importances_
feature_importance = 100.0 * (feature_importance / feature_importance.max())

indices = np.argsort(feature_importance)[-10:]
plt.barh(np.arange(10), feature_importance[indices],color='dodgerblue',alpha=.4)
plt.yticks(np.arange(10 + 0.25), np.array(X.columns)[indices])
_ = plt.xlabel('Relative importance'), plt.title('Top Ten Important Variables')

### Other Model

In [ ]:
import xgboost as xgb
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

In [ ]:
# XGBoost
clf2 = xgb.XGBClassifier(n_estimators=50, max_depth=1, 
                            learning_rate=0.01, subsample=0.8, colsample_bytree=0.3,scale_pos_weight=3.0, 
                             silent=True, nthread=-1, seed=0, missing=None,objective='binary:logistic', 
                             reg_alpha=1, reg_lambda=1, 
                             gamma=0, min_child_weight=1, 
                             max_delta_step=0,base_score=0.5)

clf2.fit(x_train, y_train)
print clf2.score(x_test, y_test)
test_pd2 = pd.DataFrame()
test_pd2['predict'] = clf2.predict(x_test)
test_pd2['label'] = y_test
print compute_ks(test_pd[['label','predict']])
print clf2.feature_importances_
# Top Ten
feature_importance = clf2.feature_importances_
feature_importance = 100.0 * (feature_importance / feature_importance.max())

indices = np.argsort(feature_importance)[-10:]
plt.barh(np.arange(10), feature_importance[indices],color='dodgerblue',alpha=.4)
plt.yticks(np.arange(10 + 0.25), np.array(X.columns)[indices])
_ = plt.xlabel('Relative importance'), plt.title('Top Ten Important Variables')

In [ ]:
# RFR
clf3 = RandomForestRegressor(n_jobs=-1, max_depth=10,random_state=0)
clf3.fit(x_train, y_train)
print clf3.score(x_test, y_test)
test_pd3 = pd.DataFrame()
test_pd3['predict'] = clf3.predict(x_test)
test_pd3['label'] = y_test
print compute_ks(test_pd[['label','predict']])
print clf3.feature_importances_
# Top Ten
feature_importance = clf3.feature_importances_
feature_importance = 100.0 * (feature_importance / feature_importance.max())

indices = np.argsort(feature_importance)[-10:]
plt.barh(np.arange(10), feature_importance[indices],color='dodgerblue',alpha=.4)
plt.yticks(np.arange(10 + 0.25), np.array(X.columns)[indices])
_ = plt.xlabel('Relative importance'), plt.title('Top Ten Important Variables')

In [ ]:
# XTR
clf4 = ExtraTreesRegressor(n_jobs=-1, max_depth=10,random_state=0)
clf4.fit(x_train, y_train)
print clf4.score(x_test, y_test)
test_pd4 = pd.DataFrame()
test_pd4['predict'] = clf4.predict(x_test)
test_pd4['label'] = y_test
print compute_ks(test_pd[['label','predict']])
print clf4.feature_importances_
# Top Ten
feature_importance = clf4.feature_importances_
feature_importance = 100.0 * (feature_importance / feature_importance.max())

indices = np.argsort(feature_importance)[-10:]
plt.barh(np.arange(10), feature_importance[indices],color='dodgerblue',alpha=.4)
plt.yticks(np.arange(10 + 0.25), np.array(X.columns)[indices])
_ = plt.xlabel('Relative importance'), plt.title('Top Ten Important Variables')

## ??:stacking